# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


/venv/ahn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-28 02:46:45.967979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787885205.986309    1203 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787885205.991982    1203 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-28 02:46:46.061393: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To ena

[2026-08-28 02:46:48,885] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/venv/ahn/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/venv/ahn/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.50it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [5]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [6]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [7]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 0.8 min
[40/168] 120 rows, 1.4 min
[60/168] 180 rows, 2.0 min
[80/168] 240 rows, 2.6 min
[100/168] 300 rows, 3.3 min
[120/168] 360 rows, 3.9 min
[140/168] 420 rows, 4.5 min
[160/168] 480 rows, 5.1 min
main sweep: 504 rows in 5.4 min


In [8]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [9]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 87674.86309523809,
    "mean_rank_residual_delta": 100328.38492063493,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.0000000000002827,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 88642.22916666667,
    "mean_rank_shuffled": 72942.9375,
    "passed": false,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 77411.31944444444,
    "mean_rank_evicted": 87674.86309523809,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout is wron

## Adding more metrics

In [10]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90737  mean_p_mem=4.352e-19
layer 18: mean_rank=96013  mean_p_mem=2.576e-07
layer 27: mean_rank=76275  mean_p_mem=6.192e-07
Paris 9 83 p_mem= 7.275948623650251e-22 p_distractor= 1.714715117563666e-21
Paris 18 83 p_mem= 2.646457986088535e-08 p_distractor= 9.625543029301298e-09
Paris 27 83 p_mem= 1.1793982821473037e-06 p_distractor= 1.7403991137143748e-07
Paris 9 90 p_mem= 2.8307047003850666e-18 p_distractor= 1.0185407078099431e-16
Paris 18 90 p_mem= 5.6501504863692986e-11 p_distractor= 7.927710571342672e-11
Paris 27 90 p_mem= 1.0555807783418913e-08 p_distractor= 1.875064326029019e-09
Paris 9 87 p_mem= 2.300567805888265e-22 p_distractor= 3.4487857767849324e-20
Paris 18 87 p_mem= 4.1683745166665176e-07 p_distractor= 1.7494253157224193e-08
Paris 27 87 p_mem= 3.3263786463066936e-06 p_distractor= 5.444185262604151e-07
Paris 9 275 p_mem= 3.849272425005347e-24 p_distractor= 8.336366502013564e-24


In [11]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90737  passes=False
layer 18: n=168 mean_rank=96013  passes=False
layer 27: n=168 mean_rank=76275  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.706  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.556  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=4.730  frac_needle>distractor=0.75  frac_pass_10x=0.17


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


In [12]:
import pandas as pd

df = pd.DataFrame(rows)

c2 = df[
    (df["layer"] == 27) &
    df["p_distractor"].notna()
].copy()

c2["ratio"] = (c2["p_mem"] + 1e-30) / (c2["p_distractor"] + 1e-30)

print("=== By needle ===")
print(
    c2.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\n=== By eviction distance ===")
print(
    c2.groupby("eviction_distance")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_index()
)

=== By needle ===
         count    median       mean
needle                             
banana      26  0.040049   0.056980
lantern     26  3.894001   5.262774
Tokyo       26  5.946284   6.408558
Paris       26  9.727367  19.082811

=== By eviction distance ===
                   count    median       mean
eviction_distance                            
-8036                  4  4.631547   3.750279
-8033                  4  4.252182   3.680604
-8029                  4  5.034098   9.206412
 83                    4  6.024562   5.436323
 87                    4  3.285810   3.184954
 90                    4  3.565137   6.480350
 267                   4  3.307607   4.097273
 275                   4  6.806819   5.843938
 277                   4  3.668272   4.346381
 527                   4  3.501555   7.148006
 532                   4  5.165074   5.071205
 539                   4  8.243987   9.691613
 1042                  4  5.926045   5.408806
 1043                  8  3.709183   7.228039


**Finding:** C2 does not fail equally for every word. J-Lens almost correctly distinguishes `Paris` from its distractor (9.73×), but performs very poorly for `banana` (0.04×). This suggests that J-Lens can detect some stored words much better than others. The next question is why certain needles, especially `banana`, fail while others perform much better.

Some rows had negative `eviction_distance` values (`-8029`, `-8033`, `-8036`).


In [13]:
neg = c2[c2["eviction_distance"] < 0]

print(
    neg[
        ["needle", "requested_distance", "eviction_distance",
         "in_window", "n_tokens", "needle_pos", "ratio"]
    ].to_string(index=False)
)

 needle  requested_distance  eviction_distance  in_window  n_tokens  needle_pos     ratio
  Paris                   0              -8029       True      8484        8449 26.725237
  Paris                   0              -8036       True      8478        8450  5.640188
  Paris                   0              -8033       True      8492        8461  5.738376
 banana                   0              -8029       True      8484        8449  0.032215
 banana                   0              -8036       True      8478        8450  0.097834
 banana                   0              -8033       True      8492        8461  0.074354
  Tokyo                   0              -8029       True      8484        8449  8.955214
  Tokyo                   0              -8036       True      8478        8450  3.956835
  Tokyo                   0              -8033       True      8492        8461  6.143698
lantern                   0              -8029       True      8484        8449  1.112981
lantern   


After inspection, these are **not errors**. All of them have:

- `requested_distance = 0`
- `in_window = True`

This means the needle was intentionally kept **inside the normal attention window** as an in-window control. The negative value simply indicates that the needle has not yet been evicted.

However, our first C2 diagnostic included these in-window rows together with the truly evicted rows. Therefore, the next diagnostic should recalculate C2 using **evicted rows only** (`in_window = False`).

In [14]:
c2_evicted = c2[c2["in_window"] == False].copy()

print("=== C2 Layer 27 — evicted rows only ===")

print(
    c2_evicted.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\nOverall median:",
      c2_evicted["ratio"].median())

=== C2 Layer 27 — evicted rows only ===
         count    median       mean
needle                             
banana      23  0.037094   0.055525
lantern     23  3.930133   5.549865
Tokyo       23  5.748870   6.415946
Paris       23  9.746041  19.915186

Overall median: 4.495314498092579


#### C2 — Evicted Rows Only

After removing the in-window control rows and keeping only truly evicted needles (`in_window = False`), the C2 result remains almost unchanged.

| Needle | Median Ratio |
|---|---:|
| banana | 0.037× |
| lantern | 3.930× |
| Tokyo | 5.749× |
| Paris | 9.746× |

Overall median ratio = **4.495×**, below the required **10×**.

**Conclusion:** The in-window rows were not responsible for the C2 failure. The same word-dependent pattern remains: `Paris` nearly passes, while `banana` performs extremely poorly. Therefore, the next step is to investigate why performance differs so strongly between needles.

In [15]:
compare = c2_evicted[
    c2_evicted["needle"].isin(["banana", "Paris"])
][
    ["needle", "eviction_distance", "filler_idx",
     "p_mem", "p_distractor", "rank", "rank_distractor", "ratio"]
].copy()

print(
    compare.sort_values(["needle", "eviction_distance"])
           .to_string(index=False)
)

needle  eviction_distance  filler_idx        p_mem  p_distractor   rank  rank_distractor     ratio
 Paris                 83           0 1.179398e-06  1.740399e-07   5698          17352.0  6.776597
 Paris                 87           2 3.326379e-06  5.444185e-07   7797          24339.0  6.109966
 Paris                 90           1 1.055581e-08  1.875064e-09  39980          69772.0  5.629571
 Paris                267           2 3.590322e-06  3.698048e-07   7177          29687.0  9.708694
 Paris                275           0 1.182175e-06  1.321255e-07   7533          25117.0  8.947361
 Paris                277           1 1.600323e-06  2.765644e-07  10310          27242.0  5.786438
 Paris                527           2 3.637167e-06  1.688676e-07   9639          53213.0 21.538569
 Paris                532           1 5.608588e-06  5.658571e-07   5969          23626.0  9.911668
 Paris                539           0 9.990758e-07  4.489314e-08   7442          36581.0 22.254533
 Paris    

#### C2 — Paris vs. Banana

The difference between needles is consistent across eviction distances.

- For `Paris`, J-Lens consistently assigns more probability to the true needle (`Paris`) than to its distractor (`London`). Some measurements exceed the 10× C2 threshold by a large margin (e.g., 21×, 34×, 43×, 69×).
- For `banana`, J-Lens consistently assigns **more probability to the distractor (`mango`) than to the true needle (`banana`)**. All inspected needle/distractor ratios are below 1.

**Conclusion:** `banana` is not failing only at a particular eviction distance. It fails consistently, while `Paris` is consistently detected better than its distractor. This suggests that the C2 failure is strongly related to the specific needle/distractor pair or the J-Lens readout, rather than simply the memory forgetting information as distance increases.

In [16]:
# Compare the actual probabilities for each needle/distractor pair
summary = (
    c2_evicted.groupby("needle")
    .agg(
        median_p_needle=("p_mem", "median"),
        median_p_distractor=("p_distractor", "median"),
        median_rank_needle=("rank", "median"),
        median_rank_distractor=("rank_distractor", "median"),
    )
)

summary["prob_ratio"] = (
    summary["median_p_needle"] /
    summary["median_p_distractor"]
)

print(summary.to_string())

         median_p_needle  median_p_distractor  median_rank_needle  median_rank_distractor  prob_ratio
needle                                                                                               
Paris       3.590322e-06         1.773017e-07              7614.0                 31474.0   20.249789
Tokyo       7.557110e-07         1.233308e-07             16955.0                 41462.0    6.127511
banana      2.833519e-10         6.734562e-09            148144.0                115796.0    0.042074
lantern     2.872485e-08         1.288016e-08             73258.0                110826.0    2.230163


#### C2 — Needle vs. Distractor Probability and Rank

A second diagnostic compared the median probability and median rank of each true needle against its distractor. This is a diagnostic only and is **not the official C2 statistic**.

The same word-dependent pattern appears in both probability and rank.

Most importantly, for `banana`:

- Median `banana` probability: 2.83e-10
- Median `mango` probability: 6.73e-09
- Median `banana` rank: 148,144
- Median `mango` rank: 115,796

Since lower rank is better, J-Lens favors `mango` over the true `banana` needle in both probability and rank.

**Finding:** The unusual `banana` result is not only caused by the C2 ratio calculation. Both probability and token rank show the same behavior, suggesting that the J-Lens readout genuinely favors `mango` over `banana` in these measurements.

In [17]:
# Diagnostic: needle vs distractor while needle is still IN the attention window.
# Uses existing rows only; does not run the model.

c2_inwindow = c2[
    (c2["in_window"] == True) &
    (c2["needle"].isin(["Paris", "banana", "Tokyo", "lantern"]))
].copy()

# Recompute the per-row ratio consistently with the earlier diagnostic.
EPS = 1e-30
c2_inwindow["ratio_check"] = (
    (c2_inwindow["p_mem"].astype(float) + EPS) /
    (c2_inwindow["p_distractor"].astype(float) + EPS)
)

summary_inwindow = (
    c2_inwindow.groupby("needle")["ratio_check"]
    .agg(["count", "median", "min", "max"])
    .sort_values("median")
)

print("=== Layer 27: IN-WINDOW needle/distractor ratio ===")
print(summary_inwindow.to_string())

=== Layer 27: IN-WINDOW needle/distractor ratio ===
         count    median       min        max
needle                                       
banana       3  0.074354  0.032215   0.097834
lantern      3  2.765988  1.112981   5.306259
Paris        3  5.738376  5.640188  26.725237
Tokyo        3  6.143698  3.956835   8.955214


#### C2 — In-Window Diagnostic

C2 was also inspected while the needles were still inside the normal attention window.

| Needle | In-Window Median Ratio |
|---|---:|
| banana | 0.074× |
| lantern | 2.766× |
| Paris | 5.738× |
| Tokyo | 6.144× |

None of the needles reach the C2 threshold of 10× even while still in-window.

Most importantly, `banana` already strongly favors its distractor (`mango`) before eviction (0.074×). Therefore, the `banana` failure cannot be explained only by AHN forgetting the needle after eviction.

**Finding:** The C2 problem appears to exist before eviction. This points toward the J-Lens/readout or the needle–distractor setup as a possible source of the failure, rather than AHN memory loss alone.

In [18]:
# C2 diagnostic: consistency of needle-vs-distractor preference
# Existing results only — no model/GPU inference.

check = c2_evicted.copy()

check["needle_wins"] = check["p_mem"] > check["p_distractor"]

summary = (
    check.groupby(["layer", "needle"])
    .agg(
        n=("needle_wins", "size"),
        needle_win_rate=("needle_wins", "mean"),
    )
)

print(summary.to_string())

                n  needle_win_rate
layer needle                      
27    Paris    23         1.000000
      Tokyo    23         1.000000
      banana   23         0.000000
      lantern  23         0.913043


In [19]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    print(
        needle, "->", n_ids, repr(tok.decode(n_ids)),
        "|",
        distractor, "->", d_ids, repr(tok.decode(d_ids))
    )

Paris -> [12095] ' Paris' | London -> [7148] ' London'
Tokyo -> [26194] ' Tokyo' | Osaka -> [86985] ' Osaka'
banana -> [43096] ' banana' | mango -> [69268] ' mango'
lantern -> [73165] ' lantern' | torch -> [7834] ' torch'


#### C2 — Pair-Specific Diagnostic

Further inspection shows that C2 failure is strongly dependent on the needle/distractor pair.

At layer 27, using only truly evicted rows:

| Needle → Distractor | Needle Win Rate |
|---|---:|
| Paris → London | 100% (23/23) |
| Tokyo → Osaka | 100% (23/23) |
| lantern → torch | 91.3% (21/23) |
| banana → mango | 0% (0/23) |

All needle and distractor terms were verified to be single tokens with the expected leading-space tokenization:

- Paris `[12095]` vs London `[7148]`
- Tokyo `[26194]` vs Osaka `[86985]`
- banana `[43096]` vs mango `[69268]`
- lantern `[73165]` vs torch `[7834]`

**Finding:** C2 is not failing uniformly. Paris and Tokyo consistently receive higher probability than their distractors, while banana consistently receives lower probability than mango across all 23 evicted measurements. Tokenization does not explain this difference.

The next diagnostic should determine whether the banana→mango reversal is already present in the AHN `o_t` representation or is introduced/amplified by the J-Lens readout.

In [20]:
# Diagnostic only:
# Compare plain logit lens vs J-Lens on the SAME AHN o_t vector.
# One banana→mango case and one Paris→London case.
# Does not modify or save experiment results.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "banana": "mango",
    "Paris": "London",
}

for needle, distractor in pairs.items():

    # Use exactly the token convention used by C2.
    needle_id = tok.encode(
        f" {needle}", add_special_tokens=False
    )[0]

    distractor_id = tok.encode(
        f" {distractor}", add_special_tokens=False
    )[0]

    # Build the same NIAH prompt used by the experiment.
    spec = ai.build_niah_prompt(
        tok,
        needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    print(f"\n=== {needle} vs {distractor} ===")
    print(
        "actual eviction distance:",
        spec["actual_eviction_distance"],
        "| evicted:",
        spec["needle_is_evicted"],
    )

    assert spec["needle_is_evicted"], (
        f"{needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt"
    ).to(bundle.model.device)

    # One forward pass. We only need AHN-on o_t.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    # SAME o_t, two different readouts.
    logits_plain = ai.readout_logits(
        o_t,
        bundle,
        lens=None,
        layer=L,
    )

    logits_jlens = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for name, logits in [
        ("PLAIN", logits_plain),
        ("J-LENS", logits_jlens),
    ]:
        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        r_n = ai.token_rank(logits, needle_id)
        r_d = ai.token_rank(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        print(
            f"{name:6s} | "
            f"p_needle={p_n:.3e} "
            f"p_dist={p_d:.3e} "
            f"ratio={ratio:.4g} | "
            f"rank_needle={r_n} "
            f"rank_dist={r_d}"
        )


=== banana vs mango ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=4.588e-10 p_dist=3.035e-08 ratio=0.01512 | rank_needle=143011 rank_dist=85872
J-LENS | p_needle=3.536e-11 p_dist=1.477e-09 ratio=0.02394 | rank_needle=148144 rank_dist=114650

=== Paris vs London ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=3.524e-07 p_dist=1.510e-08 ratio=23.33 | rank_needle=34354 rank_dist=99202
J-LENS | p_needle=9.651e-07 p_dist=4.478e-08 ratio=21.55 | rank_needle=7532 rank_dist=36528


#### C2 — Plain vs J-Lens diagnostic

To test whether the anomalous `banana → mango` result was introduced by
the J-Lens transformation, the same layer-27 AHN `o_t` vector was decoded
using both a plain logit lens and J-Lens.

At an actual eviction distance of 539 tokens:

| Pair | Plain ratio p(needle)/p(distractor) | J-Lens ratio |
|---|---:|---:|
| banana → mango | 0.015 | 0.024 |
| Paris → London | 23.33 | 21.55 |

For `banana → mango`, both readouts strongly favor the distractor.
For `Paris → London`, both strongly favor the true needle.

**Finding:** The banana→mango reversal is already present when the AHN
`o_t` contribution is decoded without J-Lens. J-Lens does not introduce
the direction of this anomaly.

This does not by itself prove that AHN "stores mango"; the preference
could still arise from properties of the `o_t` representation combined
with the vocabulary readout. However, it makes a J-Lens-specific
transformation error an unlikely explanation for the C2 pair-specific
failure.

In [21]:
# Diagnostic only:
# Is mango generally favored over banana by AHN o_t readout,
# even when the stored needle is NOT banana?

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

banana_id = tok.encode(" banana", add_special_tokens=False)[0]
mango_id  = tok.encode(" mango", add_special_tokens=False)[0]

test_needles = ["Paris", "Tokyo", "banana", "lantern"]

print("Stored needle | p(banana)/p(mango) | winner")
print("-" * 50)

for stored_needle in test_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored_needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored_needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    p_banana = ai.token_prob(logits, banana_id)
    p_mango  = ai.token_prob(logits, mango_id)

    ratio = (p_banana + 1e-30) / (p_mango + 1e-30)

    winner = "banana" if ratio > 1 else "mango"

    print(
        f"{stored_needle:12s} | "
        f"{ratio:17.6g} | {winner}"
    )

Stored needle | p(banana)/p(mango) | winner
--------------------------------------------------
Paris        |         0.0207231 | mango
Tokyo        |         0.0184562 | mango
banana       |         0.0239433 | mango
lantern      |          0.032312 | mango


#### C2 — Evidence of pair-specific baseline readout bias

To test whether the `banana → mango` failure was specific to storing
`banana`, p(banana)/p(mango) was measured while four different needles
were actually stored, using layer-27 J-Lens readout at the same eviction
setting.

| Stored needle | p(banana)/p(mango) |
|---|---:|
| Paris | 0.0207 |
| Tokyo | 0.0185 |
| banana | 0.0239 |
| lantern | 0.0323 |

`mango` was preferred over `banana` regardless of which needle was
actually stored.

**Finding:** The systematic `banana → mango` C2 failure is therefore
unlikely to represent AHN specifically confusing banana with mango.
Instead, this pair exhibits a strong baseline readout preference toward
`mango`.

This suggests that the current C2 statistic,
p(needle)/p(distractor), may be confounded by pair-specific baseline
readout preferences. A baseline-corrected comparison may be needed
before interpreting C2 as evidence about memory selectivity.

In [22]:
# Diagnostic only:
# Measure pair preference while varying the actually stored needle.
# Same layer, distance, filler, J-Lens readout for every comparison.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

stored_needles = list(pairs.keys())

# Verify every token used below is exactly one token.
pair_ids = {}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    assert len(n_ids) == 1, (needle, n_ids)
    assert len(d_ids) == 1, (distractor, d_ids)

    pair_ids[needle] = (n_ids[0], d_ids[0])


print("Stored      | Tested pair       | needle/dist ratio | winner")
print("-" * 68)

for stored in stored_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    # One forward pass per stored needle.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for needle, distractor in pairs.items():

        needle_id, distractor_id = pair_ids[needle]

        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        winner = needle if ratio > 1 else distractor

        print(
            f"{stored:11s} | "
            f"{needle:7s}/{distractor:7s} | "
            f"{ratio:17.6g} | {winner}"
        )

    print("-" * 68)

Stored      | Tested pair       | needle/dist ratio | winner
--------------------------------------------------------------------
Paris       | Paris  /London  |           21.5543 | Paris
Paris       | Tokyo  /Osaka   |           6.49267 | Tokyo
Paris       | banana /mango   |         0.0207231 | mango
Paris       | lantern/torch   |           12.5071 | lantern
--------------------------------------------------------------------
Tokyo       | Paris  /London  |           26.0682 | Paris
Tokyo       | Tokyo  /Osaka   |           7.01429 | Tokyo
Tokyo       | banana /mango   |         0.0184562 | mango
Tokyo       | lantern/torch   |           10.8719 | lantern
--------------------------------------------------------------------
banana      | Paris  /London  |            20.012 | Paris
banana      | Tokyo  /Osaka   |           6.62839 | Tokyo
banana      | banana /mango   |         0.0239433 | mango
banana      | lantern/torch   |           9.39896 | lantern
------------------------------

#### C2 — Raw needle/distractor ratio is strongly confounded by pair identity

A cross-pair diagnostic was performed at layer 27. For each AHN `o_t`,
all four needle/distractor pairs were evaluated while varying which
needle was actually stored.

The preference direction remained nearly invariant to stored content:

- Paris > London: ~15–26×
- Tokyo > Osaka: ~6.5–7×
- mango > banana: ~31–54×
- lantern > torch: ~9–12.5×

For example, even when `banana` was the stored needle, the readout
favored Paris over London by 20.0×, Tokyo over Osaka by 6.63×,
mango over banana by ~41.8×, and lantern over torch by 9.40×.

**Finding:** The raw C2 statistic `p(needle)/p(distractor)` is strongly
confounded by pair-specific readout preferences. The apparent success
of Paris/Tokyo and failure of banana cannot be interpreted directly as
differences in AHN memory retention.

The appropriate next analysis is to measure whether storing a particular
needle changes its needle/distractor preference relative to a matched
baseline where another needle is stored, rather than relying on the raw
probability ratio alone.

In [23]:
import numpy as np

# Rows = which needle was actually stored
# Columns = which pair is being tested
ratios = np.array([
    [21.5543, 6.49267, 0.0207231, 12.5071],  # stored Paris
    [26.0682, 7.01429, 0.0184562, 10.8719],  # stored Tokyo
    [20.0120, 6.62839, 0.0239433,  9.39896], # stored banana
    [15.4170, 6.53695, 0.0323120,  9.47369], # stored lantern
])

names = ["Paris", "Tokyo", "banana", "lantern"]

print("Needle   | when stored | baseline(other 3) | fold change")
print("-" * 64)

for i, name in enumerate(names):
    when_stored = ratios[i, i]

    # Baseline for this SAME pair when some other needle was stored.
    others = np.delete(ratios[:, i], i)

    # Geometric mean is appropriate because these are probability ratios.
    baseline = np.exp(np.mean(np.log(others)))

    fold_change = when_stored / baseline

    print(
        f"{name:8s} | "
        f"{when_stored:11.5g} | "
        f"{baseline:17.5g} | "
        f"{fold_change:11.4f}x"
    )

Needle   | when stored | baseline(other 3) | fold change
----------------------------------------------------------------
Paris    |      21.554 |            20.036 |      1.0758x
Tokyo    |      7.0143 |            6.5524 |      1.0705x
banana   |    0.023943 |           0.02312 |      1.0356x
lantern  |      9.4737 |            10.852 |      0.8730x


#### C2 — Baseline-corrected storage-specific effect

Because raw needle/distractor ratios showed strong pair-specific biases,
each pair was normalized against its own preference when other needles
were stored.

At layer 27 and the tested eviction setting:

| Needle | Raw ratio when stored | Baseline (other needles) | Storage-specific fold change |
|---|---:|---:|---:|
| Paris | 21.55 | 20.04 | 1.076× |
| Tokyo | 7.01 | 6.55 | 1.071× |
| banana | 0.0239 | 0.0231 | 1.036× |
| lantern | 9.47 | 10.85 | 0.873× |

Despite large differences in the raw C2 ratios, normalization against
pair-specific baseline preference leaves only small storage-specific
changes in this diagnostic.

**Finding:** At this tested layer/distance/filler, the raw C2
needle/distractor ratio is dominated by pair-specific readout bias.
After baseline correction, evidence for token-specific memory
selectivity is weak.

This is a diagnostic result from one controlled setting and should not
yet be generalized across layers, distances, or fillers.

In [24]:
print("Distances:", EXP.get("distances"))
print("Fillers:", EXP.get("filler_indices"))

print("\nEXP:")
for k, v in EXP.items():
    print(f"{k}: {v}")

Distances: None
Fillers: None

EXP:
layers: [9, 18, 27]
eviction_distances: [64, 256, 512, 1024, 2048, 4096, 8192]
needle_candidates: ['Paris', 'banana', 'Tokyo', 'violin', 'cinnamon', 'harbour', 'lantern', 'sapphire', 'meadow', 'trumpet']
n_filler_variants: 3
use_jlens: True
jlens_path: results/run_3b_gdn/jlens_qwen25_3b.pt
lens_validated: False


In [25]:
# Inspection only — existing C2 rows.
# No inference and no modification of results.

c2_rows = [
    r for r in main
    if "p_distractor" in r
]

print("Total C2 rows:", len(c2_rows))
print("Needles:", sorted(set(r["needle"] for r in c2_rows)))
print("Layers:", sorted(set(r["layer"] for r in c2_rows)))
print(
    "Requested distances:",
    sorted(set(r["requested_distance"] for r in c2_rows))
)
print(
    "Filler indices:",
    sorted(set(r["filler_idx"] for r in c2_rows))
)

print("\nRows per layer / needle:")
from collections import Counter

counts = Counter(
    (r["layer"], r["needle"])
    for r in c2_rows
)

for key in sorted(counts):
    print(key, counts[key])

Total C2 rows: 252
Needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [9, 18, 27]
Requested distances: [64, 256, 512, 1024, 2048, 4096, 8192]
Filler indices: [0, 1, 2]

Rows per layer / needle:
(9, 'Paris') 21
(9, 'Tokyo') 21
(9, 'banana') 21
(9, 'lantern') 21
(18, 'Paris') 21
(18, 'Tokyo') 21
(18, 'banana') 21
(18, 'lantern') 21
(27, 'Paris') 21
(27, 'Tokyo') 21
(27, 'banana') 21
(27, 'lantern') 21


In [26]:
# FULL C2 BASELINE-BIAS DIAGNOSTIC
# --------------------------------
# 4 needles × 7 distances × 3 fillers = 84 forward passes.
# Each forward pass captures layers 9, 18, 27 together.
#
# Diagnostic only:
# - does NOT modify `main`
# - does NOT overwrite official result files
# - stores output in `c2_bias_rows`

import numpy as np
import pandas as pd

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# --------------------------------------------------
# 1. Verify tokenization before spending GPU compute
# --------------------------------------------------

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (n_ids[0], d_ids[0])


# --------------------------------------------------
# 2. Controlled sweep
# --------------------------------------------------

c2_bias_rows = []

total = len(PAIRS) * len(DISTANCES) * len(list(FILLERS))
done = 0

for stored_needle in PAIRS:

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                stored_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            # Match the official experiment's validity conditions.
            if not spec["ahn_will_activate"]:
                continue

            if not spec["needle_is_evicted"]:
                continue

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            # ONE model forward pass captures all three layers.
            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                # Evaluate ALL four pairs from this SAME o_t.
                for tested_needle, distractor in PAIRS.items():

                    needle_id, distractor_id = pair_ids[tested_needle]

                    p_n = ai.token_prob(
                        logits,
                        needle_id,
                    )

                    p_d = ai.token_prob(
                        logits,
                        distractor_id,
                    )

                    # 1e-30 only prevents division by zero.
                    # Unlike the official 1e-12, it does not dominate
                    # the tiny probabilities observed in these rows.
                    ratio = (
                        (p_n + 1e-30) /
                        (p_d + 1e-30)
                    )

                    c2_bias_rows.append({
                        "stored_needle": stored_needle,
                        "tested_needle": tested_needle,
                        "distractor": distractor,
                        "layer": L,
                        "requested_distance": requested_distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_needle": p_n,
                        "p_distractor": p_d,
                        "ratio": ratio,
                    })

            done += 1

            if done % 10 == 0 or done == total:
                print(
                    f"{done}/{total} forward passes completed"
                )


# --------------------------------------------------
# 3. Sanity checks
# --------------------------------------------------

df_bias = pd.DataFrame(c2_bias_rows)

print("\n=== SWEEP COMPLETE ===")
print("Forward passes expected:", total)
print("Diagnostic rows:", len(df_bias))

print(
    "Stored needles:",
    sorted(df_bias["stored_needle"].unique())
)

print(
    "Tested needles:",
    sorted(df_bias["tested_needle"].unique())
)

print(
    "Layers:",
    sorted(df_bias["layer"].unique())
)

print(
    "Requested distances:",
    sorted(df_bias["requested_distance"].unique())
)

print(
    "Fillers:",
    sorted(df_bias["filler_idx"].unique())
)

print("\nRows by layer / stored needle / tested needle:")

print(
    df_bias
    .groupby(
        ["layer", "stored_needle", "tested_needle"]
    )
    .size()
    .to_string()
)

10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

=== SWEEP COMPLETE ===
Forward passes expected: 84
Diagnostic rows: 1008
Stored needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Tested needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [9, 18, 27]
Requested distances: [64, 256, 512, 1024, 2048, 4096, 8192]
Fillers: [0, 1, 2]

Rows by layer / stored needle / tested needle:
layer  stored_needle  tested_needle
9      Paris          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       Tokyo          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       banana   

In [27]:
# FULL C2 baseline-corrected analysis
# Uses df_bias already generated.
# No model inference. Does not modify official results.

import numpy as np
import pandas as pd

# Work in log-ratio space:
# log[p(needle)/p(distractor)]
#
# For every exact:
#   layer × distance × filler × tested pair
#
# compare:
#   ratio when THAT needle was stored
# versus
#   mean log-ratio when the other 3 needles were stored.

work = df_bias.copy()

# Numerical guard only.
EPS = 1e-30

work["log_ratio"] = np.log(
    (work["p_needle"] + EPS) /
    (work["p_distractor"] + EPS)
)

corrected = []

group_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
]

for keys, g in work.groupby(group_cols):

    layer, distance, filler, tested = keys

    # The row where the tested needle is actually the stored needle.
    target = g[g["stored_needle"] == tested]

    # Matched baseline: SAME layer/distance/filler/pair,
    # but one of the other three needles was stored.
    baseline = g[g["stored_needle"] != tested]

    assert len(target) == 1, (keys, len(target))
    assert len(baseline) == 3, (keys, len(baseline))

    target_log = float(target["log_ratio"].iloc[0])
    baseline_log = float(baseline["log_ratio"].mean())

    delta_log = target_log - baseline_log

    corrected.append({
        "layer": layer,
        "requested_distance": distance,
        "filler_idx": filler,
        "needle": tested,
        "target_ratio": float(np.exp(target_log)),
        "baseline_ratio": float(np.exp(baseline_log)),
        "corrected_fold": float(np.exp(delta_log)),
        "delta_log_ratio": delta_log,
    })

df_corrected = pd.DataFrame(corrected)

# Expected:
# 3 layers × 7 distances × 3 fillers × 4 needles = 252
assert len(df_corrected) == 252

print("Corrected observations:", len(df_corrected))

print("\n=== FULL BASELINE-CORRECTED C2 ===")

summary = (
    df_corrected
    .groupby(["layer", "needle"])
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(summary.to_string())

print("\n=== POOLED BY LAYER ===")

layer_summary = (
    df_corrected
    .groupby("layer")
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(layer_summary.to_string())

Corrected observations: 252

=== FULL BASELINE-CORRECTED C2 ===
                n  median_corrected_fold  geometric_mean_fold  frac_above_1
layer needle                                                               
9     Paris    21               0.978864             0.978119      0.285714
      Tokyo    21               0.995770             1.005828      0.476190
      banana   21               1.004803             1.009428      0.619048
      lantern  21               0.987792             0.968071      0.380952
18    Paris    21               0.986777             0.994758      0.476190
      Tokyo    21               1.000091             1.024329      0.523810
      banana   21               1.012495             1.031188      0.571429
      lantern  21               1.169215             1.141920      0.714286
27    Paris    21               1.072769             1.083832      0.809524
      Tokyo    21               1.040508             1.034738      0.666667
      banana   21       

#### C2 — Full matched baseline-corrected analysis

The pair-specific readout-bias diagnostic was extended across the full
C2 design: 3 layers × 7 eviction distances × 3 filler variants ×
4 needle/distractor pairs (252 matched corrected observations).

For every layer × distance × filler × tested-pair condition, the
needle/distractor log-probability ratio when the tested needle was
actually stored was compared with the same pair's mean log-ratio when
one of the other three needles was stored.

Pooled results:

| Layer | Median corrected fold | Geometric mean fold | Fraction > 1 |
|---|---:|---:|---:|
| 9  | 0.995 | 0.990 | 44.0% |
| 18 | 1.018 | 1.047 | 57.1% |
| 27 | 1.035 | 0.996 | 54.8% |

After controlling for pair-specific baseline preference, the pooled
storage-specific effect is close to 1× at all three layers.

**Finding:** The large raw differences observed in C2 are strongly
confounded by pair-specific readout preferences. Across the full tested
design, matching each pair against its own baseline leaves little
pooled evidence that storing the tested needle substantially increases
its needle-vs-distractor readout ratio.

This result concerns the validity/interpretability of the current C2
readout statistic. It does not establish that AHN contains no
token-specific information, because such information may not be
recoverable by the current J-Lens/vocabulary readout.

In [29]:
# Leakage check: inspect whether C2 target/distractor words
# accidentally appear in prompts where they should not.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

for stored in ["Paris", "Tokyo", "banana", "lantern"]:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=512,
        in_window=False,
        filler_idx=0,
    )

    prompt = spec["prompt"]

    print(f"\n=== STORED: {stored} ===")

    for word in WORDS:
        count = prompt.lower().count(word.lower())

        if count:
            print(f"{word:8s}: {count}")


=== STORED: Paris ===
Paris   : 1

=== STORED: Tokyo ===
Tokyo   : 1

=== STORED: banana ===
banana  : 1

=== STORED: lantern ===
lantern : 1


In [31]:
# FULL C2 prompt-leakage check
# 4 needles × 7 distances × 3 fillers = 84 prompts
# NO model inference / NO GPU / modifies nothing.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

STORED = ["Paris", "Tokyo", "banana", "lantern"]

leaks = []
checked = 0

for stored in STORED:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tok,
                stored,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            prompt_lower = spec["prompt"].lower()
            checked += 1

            for word in WORDS:
                count = prompt_lower.count(word.lower())

                # Intended stored needle should appear exactly once.
                if word == stored:
                    if count != 1:
                        leaks.append({
                            "stored": stored,
                            "distance": distance,
                            "filler": filler_idx,
                            "word": word,
                            "count": count,
                            "problem": "stored needle count != 1",
                        })

                # Every other C2 word should be absent.
                elif count != 0:
                    leaks.append({
                        "stored": stored,
                        "distance": distance,
                        "filler": filler_idx,
                        "word": word,
                        "count": count,
                        "problem": "unexpected word in prompt",
                    })

print("Prompts checked:", checked)
print("Problems found:", len(leaks))

if leaks:
    for x in leaks:
        print(x)
else:
    print("PASS: no C2 target/distractor contamination detected.")

Prompts checked: 84
Problems found: 0
PASS: no C2 target/distractor contamination detected.


#### C2 — Prompt contamination check

All 84 prompts used in the full C2 diagnostic
(4 needles × 7 distances × 3 filler variants) were checked for
accidental occurrences of all C2 needle and distractor words.

Each prompt contained its intended stored needle exactly once and
contained none of the other tested needles or distractors.

- Prompts checked: 84
- Contamination cases: 0

**Finding:** The observed pair-specific C2 readout preferences cannot
be explained by accidental target/distractor word contamination in the
generated prompts.

This rules out this specific form of prompt-level leakage, but does not
rule out every possible source of experimental bias or leakage.

In [32]:
# C2 baseline-corrected bootstrap confidence intervals
# -----------------------------------------------------
# Uses df_corrected from the previous analysis.
# NO GPU inference.
# Does NOT modify official results.

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)
N_BOOT = 10_000

results = []

for (layer, needle), g in df_corrected.groupby(["layer", "needle"]):

    # Analyze log fold-change because:
    #   0 = no effect
    # and exp(0) = 1x fold-change.
    x = g["delta_log_ratio"].to_numpy()

    n = len(x)
    assert n == 21

    boot_means = np.empty(N_BOOT)

    for b in range(N_BOOT):
        sample = RNG.choice(x, size=n, replace=True)
        boot_means[b] = sample.mean()

    mean_log = x.mean()

    lo_log, hi_log = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    results.append({
        "layer": layer,
        "needle": needle,
        "n": n,
        "geometric_mean_fold": np.exp(mean_log),
        "ci_low": np.exp(lo_log),
        "ci_high": np.exp(hi_log),
        "ci_contains_1":
            bool(lo_log <= 0 <= hi_log),
    })


ci_by_needle = pd.DataFrame(results)

print("=== 95% BOOTSTRAP CI — BY LAYER / NEEDLE ===")

print(
    ci_by_needle
    .sort_values(["layer", "needle"])
    .to_string(index=False)
)


# -----------------------------------------------------
# Pooled layer analysis
# -----------------------------------------------------

pooled = []

for layer, g in df_corrected.groupby("layer"):

    x = g["delta_log_ratio"].to_numpy()

    n = len(x)
    assert n == 84

    boot_means = np.empty(N_BOOT)

    for b in range(N_BOOT):
        sample = RNG.choice(x, size=n, replace=True)
        boot_means[b] = sample.mean()

    mean_log = x.mean()

    lo_log, hi_log = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    pooled.append({
        "layer": layer,
        "n": n,
        "geometric_mean_fold": np.exp(mean_log),
        "ci_low": np.exp(lo_log),
        "ci_high": np.exp(hi_log),
        "ci_contains_1":
            bool(lo_log <= 0 <= hi_log),
    })


ci_by_layer = pd.DataFrame(pooled)

print("\n=== 95% BOOTSTRAP CI — POOLED BY LAYER ===")

print(
    ci_by_layer
    .sort_values("layer")
    .to_string(index=False)
)

=== 95% BOOTSTRAP CI — BY LAYER / NEEDLE ===
 layer  needle  n  geometric_mean_fold   ci_low  ci_high  ci_contains_1
     9   Paris 21             0.978119 0.959631 0.997649          False
     9   Tokyo 21             1.005828 0.981315 1.030683           True
     9  banana 21             1.009428 0.996430 1.023937           True
     9 lantern 21             0.968071 0.929448 1.001721           True
    18   Paris 21             0.994758 0.924483 1.063699           True
    18   Tokyo 21             1.024329 0.974776 1.080188           True
    18  banana 21             1.031188 0.992157 1.075123           True
    18 lantern 21             1.141920 1.058423 1.232767          False
    27   Paris 21             1.083832 1.023865 1.147952          False
    27   Tokyo 21             1.034738 0.983009 1.087171           True
    27  banana 21             1.002111 0.922157 1.091639           True
    27 lantern 21             0.875384 0.799225 0.958143          False

=== 95% BOOTSTRAP 

In [33]:
# C2 — matched/clustered statistical validation
# ----------------------------------------------
# NO GPU.
# Uses df_corrected only.
#
# Tests:
# 1. Each layer × needle: 21 matched distance×filler conditions.
# 2. Each layer pooled: first average the 4 needles WITHIN each
#    distance×filler condition -> 21 independent condition-level values.
# 3. Bootstrap 95% CI for geometric-mean fold change.
# 4. Two-sided sign-flip permutation test for mean log-fold = 0.
# 5. Holm correction for multiple comparisons.

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_BOOT = 20_000
N_PERM = 100_000


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

d = df_corrected.copy()

d["condition"] = list(zip(
    d["requested_distance"],
    d["filler_idx"]
))

assert len(d) == 252
assert d["condition"].nunique() == 21

# Every layer × condition should contain exactly 4 needles.
counts = (
    d.groupby(["layer", "condition"])
     .size()
)

assert (counts == 4).all(), counts[counts != 4]


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def bootstrap_mean_log_ci(x, n_boot=N_BOOT):
    """
    Bootstrap the mean log-fold.
    Returned values are exponentiated, so they are
    geometric-mean fold changes.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    idx = RNG.integers(
        0, n,
        size=(n_boot, n)
    )

    boot_means = x[idx].mean(axis=1)

    lo, hi = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return (
        float(np.exp(x.mean())),
        float(np.exp(lo)),
        float(np.exp(hi)),
    )


def signflip_pvalue(x, n_perm=N_PERM):
    """
    Two-sided matched sign-flip permutation test.

    H0: mean log-fold = 0.

    The sign of each matched condition is randomly flipped.
    """
    x = np.asarray(x, dtype=float)

    observed = abs(x.mean())

    n = len(x)

    # Generate +/-1 signs.
    signs = RNG.choice(
        np.array([-1.0, 1.0]),
        size=(n_perm, n)
    )

    permuted = (signs * x).mean(axis=1)

    # +1 correction avoids p=0 from finite Monte Carlo sampling.
    p = (
        np.sum(np.abs(permuted) >= observed) + 1
    ) / (n_perm + 1)

    return float(p)


def holm_adjust(pvalues):
    """
    Holm family-wise error correction.
    """
    p = np.asarray(pvalues, dtype=float)

    order = np.argsort(p)
    adjusted = np.empty_like(p)

    running_max = 0.0
    m = len(p)

    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


# ==================================================
# A. LAYER × NEEDLE TESTS
# ==================================================

needle_results = []

for (layer, needle), g in d.groupby(
    ["layer", "needle"]
):

    # Exactly one matched observation per
    # distance × filler condition.
    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["delta_log_ratio"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    needle_results.append({
        "layer": layer,
        "needle": needle,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


needle_stats = pd.DataFrame(needle_results)

# Correct across all 12 layer×needle tests.
needle_stats["p_holm"] = holm_adjust(
    needle_stats["p_raw"].to_numpy()
)

needle_stats["significant_holm_005"] = (
    needle_stats["p_holm"] < 0.05
)


print(
    "=== MATCHED TEST — LAYER × NEEDLE "
    "(Holm corrected across 12 tests) ==="
)

print(
    needle_stats
    .sort_values(["layer", "needle"])
    .to_string(index=False)
)


# ==================================================
# B. POOLED LAYER TESTS
# ==================================================
#
# IMPORTANT:
# Do NOT treat 84 rows as independent.
#
# Within every layer × distance × filler cluster,
# first average the four needle effects.
#
# This gives 21 condition-level observations/layer.

clustered = (
    d.groupby([
        "layer",
        "requested_distance",
        "filler_idx"
    ])["delta_log_ratio"]
    .mean()
    .reset_index(name="cluster_mean_log")
)

layer_results = []

for layer, g in clustered.groupby("layer"):

    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["cluster_mean_log"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    layer_results.append({
        "layer": layer,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


layer_stats = pd.DataFrame(layer_results)

# Correct across the 3 pooled layer tests.
layer_stats["p_holm"] = holm_adjust(
    layer_stats["p_raw"].to_numpy()
)

layer_stats["significant_holm_005"] = (
    layer_stats["p_holm"] < 0.05
)


print(
    "\n=== MATCHED/CLUSTERED TEST — POOLED BY LAYER "
    "(Holm corrected across 3 tests) ==="
)

print(
    layer_stats
    .sort_values("layer")
    .to_string(index=False)
)

=== MATCHED TEST — LAYER × NEEDLE (Holm corrected across 12 tests) ===
 layer  needle  n_conditions  geometric_mean_fold   ci_low  ci_high    p_raw   p_holm  significant_holm_005
     9   Paris            21             0.978119 0.959761 0.997678 0.041290 0.371606                 False
     9   Tokyo            21             1.005828 0.980753 1.030749 0.657133 1.000000                 False
     9  banana            21             1.009428 0.996424 1.023833 0.211848 1.000000                 False
     9 lantern            21             0.968071 0.930381 1.001537 0.114829 0.918631                 False
    18   Paris            21             0.994758 0.925126 1.063679 0.885371 1.000000                 False
    18   Tokyo            21             1.024329 0.975596 1.080388 0.384686 1.000000                 False
    18  banana            21             1.031188 0.992251 1.075768 0.167618 1.000000                 False
    18 lantern            21             1.141920 1.059843 1.2318

## C2 Follow-up Finding: Word Bias and a Small Memory Signal at Layer 18

### Why we investigated C2

The original C2 control checks whether the word stored in AHN memory gets a much higher probability than a distractor word.

For example:

- Paris should beat London
- Tokyo should beat Osaka
- banana should beat mango
- lantern should beat torch

The original pass requirement was that the stored word should have at least **10× higher probability** than its distractor.

C2 failed this test.

---

### 1. We found that different word pairs have very different natural biases

At first, the results looked like AHN remembered some words much better than others.

For example:

- Paris almost always beat London.
- Tokyo almost always beat Osaka.
- banana never beat mango.

However, we tested the same word pairs when **different words were actually stored**.

The preference stayed almost the same.

For example, even when `banana` was stored:

- Paris still beat London by about 20×.
- Tokyo still beat Osaka by about 6.6×.
- mango still beat banana by about 42×.
- lantern still beat torch by about 9.4×.

This means that a large part of the original C2 score depends on the **word pair itself**, not just what AHN remembered.

In simple terms, our measuring tool is already "tilted" toward some words before we try to measure memory.

---

### 2. We checked for prompt leakage

We checked all:

**4 needles × 7 distances × 3 fillers = 84 prompts**

Each prompt contained its intended stored word exactly once.

None of the other tested needles or distractors accidentally appeared in the prompt.

Results:

- Prompts checked: **84**
- Contamination found: **0**

Therefore, the word-pair bias is **not explained by the tested words accidentally appearing elsewhere in the prompts**.

This only rules out this specific type of prompt contamination. It does not rule out every possible source of bias.

---

### 3. We corrected for the word-pair bias

Instead of asking only:

> "Does Paris beat London?"

we asked:

> "Does Paris beat London MORE when Paris is actually stored than when another word is stored?"

We did this for every word pair, layer, distance, and filler.

After this correction, most of the very large differences disappeared.

Pooled corrected effects:

| Layer | Memory-specific effect |
|---|---:|
| 9 | 0.990× |
| 18 | 1.047× |
| 27 | 0.996× |

A value near **1×** means that storing the correct word produced little change compared with the baseline.

---

### 4. Statistical test

We used a matched/clustered test across the 21 distance × filler conditions and applied Holm correction for multiple comparisons.

| Layer | Corrected effect | 95% CI | Holm-adjusted p | Result |
|---|---:|---:|---:|---|
| 9 | 0.990× | 0.979–1.001 | 0.203 | Not significant |
| 18 | **1.047×** | **1.020–1.075** | **0.0086** | **Significant** |
| 27 | 0.996× | 0.975–1.018 | 0.718 | Not significant |

### Main finding

After correcting for the natural bias between word pairs:

- Layer 9 does not show a reliable memory-specific signal.
- **Layer 18 shows a small but statistically reliable memory-specific signal.**
- Layer 27 does not show a reliable memory-specific signal.

The effect at layer 18 is approximately **4.7%**, which is much smaller than the original C2 requirement of **10×**.

---

### Simple interpretation

The original C2 test made some words look much better remembered than others.

Our follow-up analysis shows that much of this difference comes from the **readout already preferring certain words over their distractors**, rather than AHN necessarily remembering those words better.

After correcting for this bias, we still find a **small but statistically reliable signal at layer 18** about which word was actually stored.

This suggests that some information about the stored word may be recoverable from the middle AHN layer, even though the original C2 test fails.

### Important limitation

We should **not** conclude that:

> "AHN stores memory mainly at layer 18."

Our experiment only measures information that can be recovered using the current J-Lens/vocabulary readout.

The J-Lens also does not pass the original Table 3 validation, so this result should be reported as evidence from an **unvalidated J-Lens readout** until that issue is resolved.